<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Propensity scores, matching and weighting

**[Propensity scores, matching and weighting](https://learning.nextia-ai.com/courses/experiments/m06/propensity-scores-matching-and-weighting/)** · Experiments and Causal Impact: Did Our Change Work? · Module 6, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** estimate each customer's chance of using the recommendation block (the propensity score), match and weight the customers with it, check balance and overlap, and compare the answers with the simulator's truth.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Regression adjustment and the six confounders, from [Regression adjustment](https://learning.nextia-ai.com/courses/experiments/m06/regression-adjustment/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m06/propensity-scores-matching-and-weighting/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M06-L02-propensity-scores-matching-and-weighting/propensity-scores-matching-and-weighting-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "logs/customers.csv.gz": "0e9bd54ac28ef98bf94f6d1d0abe1bf0f8e51f67384d7b08c94b0b9310ab320a",
    "logs/truth.json": "8e2cd0eda8b31c1fba87687dfaf14b598d24921cf38734e8d18a21d8126b009f"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The propensity score

A logistic regression of `used_block` on the six confounders gives each customer a chance to use the block: the **propensity score**. As in the project, the counts and amounts enter as `log(1 + x)`, so a few very big customers do not dominate.

In [ ]:
import statsmodels.api as sm

logs = pd.read_csv("data/logs/customers.csv.gz")
logs["desktop"] = (logs.device == "desktop").astype(int)
CONFOUNDERS = ["pre_spend", "pre_orders", "visits", "tenure_days", "new_customer", "desktop"]
LOGGED = ("pre_spend", "pre_orders", "visits", "tenure_days")

def propensity(columns):
    x = logs[columns].astype(float).copy()
    for c in columns:
        if c in LOGGED:
            x[c] = np.log1p(x[c])
    x = sm.add_constant(x)
    return sm.Logit(logs.used_block, x).fit(disp=0).predict(x).to_numpy()

ps = propensity(CONFOUNDERS)
users = logs.used_block.to_numpy() == 1
print(f"users: {ps[users].min():.3f} to {ps[users].max():.3f}; non-users: {ps[~users].min():.3f} to {ps[~users].max():.3f}")

> **Check.** You should see `users: 0.191 to 0.505; non-users: 0.191 to 0.508`.

Both groups cover the same range: every kind of user has non-users of the same kind. This is **overlap**.

## 2. Balance before adjustment

The **standardised mean difference** (SMD) of a column: the users' mean minus the non-users' mean, divided by the column's typical spread. Below 0.1 counts as balanced.

In [ ]:
def smd(columns, w=None):
    w = np.ones(len(logs)) if w is None else w
    out = {}
    for c in columns:
        x = logs[c].to_numpy(float)
        m1, m0 = np.average(x[users], weights=w[users]), np.average(x[~users], weights=w[~users])
        out[c] = round(float((m1 - m0) / np.sqrt((x[users].var() + x[~users].var()) / 2)), 2) + 0.0  # + 0.0: no "-0.0"
    return out

print("before:", smd(CONFOUNDERS))

> **Check.** You should see `before: {'pre_spend': 0.08, 'pre_orders': 0.1, 'visits': 0.09, 'tenure_days': 0.14, 'new_customer': -0.15, 'desktop': 0.15}`.

## 3. Weighting

Each user counts 1 / score, and each non-user 1 / (1 − score). A user whom the model did not expect counts a lot; so does a non-user who looked like a user. The weighted groups then look like everyone.

In [ ]:
y = logs.revenue.to_numpy()
w = np.where(users, 1 / ps, 1 / (1 - ps))
ipw = (w[users] * y[users]).sum() / w[users].sum() - (w[~users] * y[~users]).sum() / w[~users].sum()
print(f"weighting: {ipw:+.2f} dollars (ATE); truth {truth('logs')['ate']['effect']:+.2f}")
print("after weighting:", smd(CONFOUNDERS, w))

> **Check.** You should see `weighting: +2.87 dollars (ATE); truth +2.37` and `after weighting: {'pre_spend': -0.01, 'pre_orders': 0.0, 'visits': 0.0, 'tenure_days': 0.0, 'new_customer': 0.0, 'desktop': 0.0}`.

Every confounder is now balanced. The estimate is still above the truth: the garden is not in the log, so the weights cannot balance it.

## 4. Matching

Each user gets the non-user with the closest score (on the log-odds scale), within a caliper of 0.05 standard deviations. A non-user can be used more than once. The mean difference of the pairs estimates the effect **on the users** (ATT).

In [ ]:
logit = np.log(ps / (1 - ps))
lt, lc = logit[users], logit[~users]
order = np.argsort(lc)
pos = np.clip(np.searchsorted(lc[order], lt), 1, len(lc) - 1)
left, right = order[pos - 1], order[pos]
best = np.where(np.abs(lc[left] - lt) <= np.abs(lc[right] - lt), left, right)
ok = np.abs(lc[best] - lt) <= 0.05 * logit.std()
pairs = y[users][ok] - y[~users][best[ok]]
print(f"matching: {pairs.mean():+.2f} dollars (ATT), {ok.sum():,} users matched to {len(np.unique(best[ok])):,} non-users; truth {truth('logs')['att']['effect']:+.2f}")

> **Check.** You should see `matching: +4.43 dollars (ATT), 13,338 users matched to 7,994 non-users; truth +2.73`.

## 5. Your turn: a mediator breaks overlap

`pages_viewed` is a mediator: the block adds pages. Put it in the propensity model by mistake.

> **Your turn.** Compute the propensity scores with the six confounders **and** `pages_viewed`, then the weighting estimate as in section 3. Put it, rounded to 2 decimals, in `ipw_mediator`. Look at the largest score too.

In [ ]:
ps_m = propensity(CONFOUNDERS + ["pages_viewed"])
w_m = np.where(users, 1 / ps_m, 1 / (1 - ps_m))
ipw_mediator = round(float((w_m[users] * y[users]).sum() / w_m[users].sum()
                           - (w_m[~users] * y[~users]).sum() / w_m[~users].sum()), 2)
print(f"largest score {ps_m.max():.4f}; largest weight {w_m.max():,.0f}; weighting {ipw_mediator:+.2f}")

In [ ]:
expect('ipw_mediator', ipw_mediator, -282.02)

Pages predict block use almost perfectly for some customers, so their scores reach 1.0. A non-user with a score near 1 gets a weight of 1 / (1 − score): thousands. A handful of customers then decide the whole answer, and it is absurd.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m06/propensity-scores-matching-and-weighting/) has the balance and overlap charts and the knowledge checks that count toward your certificate. In the project, `xp adjust` prints all three estimates, the overlap and the balance; `xp adjust --add pages_viewed` shows the broken overlap.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Test your own answer](https://learning.nextia-ai.com/courses/experiments/m06/test-your-own-answer/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m06/propensity-scores-matching-and-weighting/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.